# Fresh-C4 Qwen3-8B and Mistral-7B reproduction

This notebook regenerates the multi-billion-parameter MLP results reported in the public reproducibility repository.

It runs only the successful reported scope:

- Qwen3-8B at relative depths approximately 0.20, 0.40, 0.60, and 0.743.
- Mistral-7B-v0.3 at relative depths approximately 0.194, 0.387, 0.613, and 0.742.
- Full receiver-specificity, dose, future-propagation, synthetic-direction, and local donor-bank audits at the approximately 0.40 depth.

The frozen protocol uses 64 fresh C4 contexts, all 4032 directed non-self MLP transplants per depth, and centered finite differences with epsilon 0.075.

The original reported run used Kaggle GPU T4 x2 in FP16. No attention-output or 14B experiments are part of this reproduction notebook.


In [ ]:

# Kaggle: keep the preinstalled scientific stack. Only install model/data packages.
%pip -q install "transformers>=4.55,<5" "accelerate>=1.6,<2" "datasets>=3.5,<4" "huggingface_hub>=0.34,<1"


In [ ]:

# ============================================================
# CONFIG — FROZEN T4x2 PAPER-GRADE PROTOCOL
# ============================================================

import os, gc, re, json, math, time, random, hashlib, shutil, zipfile, warnings, traceback
from dataclasses import dataclass, asdict
from collections import defaultdict

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from scipy.stats import spearmanr, wilcoxon, binomtest
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
torch.set_grad_enabled(False)

EXPERIMENT_VERSION = "NEWINML_FRESH_C4_REPRO_V1"
OUT_ROOT = f"/kaggle/working/{EXPERIMENT_VERSION}"
MODEL_CACHE = "/kaggle/working/hf_model_cache"
DATA_CACHE = "/kaggle/working/hf_dataset_cache"
OFFLOAD_DIR = "/kaggle/working/hf_offload"
CHECKPOINT_ZIP = f"/kaggle/working/{EXPERIMENT_VERSION}_CHECKPOINT.zip"

for p in [OUT_ROOT, MODEL_CACHE, DATA_CACHE, OFFLOAD_DIR]:
    os.makedirs(p, exist_ok=True)

os.environ["HF_HOME"] = MODEL_CACHE
os.environ["HF_DATASETS_CACHE"] = DATA_CACHE

@dataclass
class CFG:
    seed: int = 260825

    n_contexts_core: int = 64
    raw_pool_size: int = 128
    core_tokens: int = 64
    future_tokens: int = 32

    stable_eps: float = 0.075
    mlp_source_eps: float = 0.05
    default_pair_batch: int = 4

    eps_grid: tuple = (0.04, 0.05, 0.075, 0.10)
    eps_pairs: int = 128
    eps_rank_gate: float = 0.95

    wrong_pairs: int = 256
    wrong_receivers_per_pair: int = 8
    receiver_gap_gate: float = 0.15
    correct_receiver_rho_gate: float = 0.75

    residual_rho_gate: float = 0.30
    bootstrap_reps: int = 600

    dose_pairs: int = 256
    doses: tuple = (0.10, 0.25, 0.50, 0.75, 1.00)

    future_pairs: int = 128
    future_horizons: tuple = (1, 4, 8, 16, 32)
    future_batch: int = 1

    control_pairs: int = 96
    rotation_degrees: tuple = (30, 60, 90)

    candidate_sizes: tuple = (2, 4, 8, 16, 32)
    selection_alpha: float = 0.05

    n_contexts_attention: int = 32

    require_t4x2: bool = True
    allow_cpu_offload_14b: bool = True
    gpu_reserve_mib: int = 1024
    cpu_offload_cap_gib: int = 20
    delete_model_cache_after_each_model: bool = True
    resume: bool = True

cfg = CFG()

MODEL_SPECS = [
    dict(
        model_id="Qwen/Qwen3-8B",
        params_b=8.191,
        role="hero_posttrained",
        depths=[0.20, 0.40, 0.60, 0.75],
        full_extras_depth=0.40,
        run_attention=False,
        enabled=True,
        gated=False,
        t4_pair_batch=4,
    ),
    dict(
        model_id="mistralai/Mistral-7B-v0.3",
        params_b=7.248,
        role="cross_family",
        depths=[0.20, 0.40, 0.60, 0.75],
        full_extras_depth=0.40,
        run_attention=False,
        enabled=True,
        gated=False,
        t4_pair_batch=4,
    ),
]

assert torch.cuda.is_available(), "Kaggle accelerator is OFF. Select GPU T4 x2."

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
gpu_mem_gib = [
    torch.cuda.get_device_properties(i).total_memory / 2**30
    for i in range(torch.cuda.device_count())
]

print("Experiment:", EXPERIMENT_VERSION)
print("CUDA devices:", torch.cuda.device_count())
for i, (name, mem) in enumerate(zip(gpu_names, gpu_mem_gib)):
    print(f"  GPU {i}: {name} | {mem:.2f} GiB")

is_t4x2 = (
    torch.cuda.device_count() == 2 and
    all("T4" in n.upper() for n in gpu_names)
)

if cfg.require_t4x2:
    assert is_t4x2, (
        "This notebook is optimized for Kaggle T4 x2. "
        f"Detected: {gpu_names}. Change Accelerator to GPU T4 x2 and restart."
    )

MODEL_DTYPE = torch.float16
print("\nT4x2 preflight: PASS")
print("MODEL_DTYPE:", MODEL_DTYPE)

print("\nFrozen model order:")
for spec in MODEL_SPECS:
    if spec["enabled"]:
        print(
            f" - {spec['model_id']:28s} "
            f"| depths={spec['depths']} "
            f"| pair_batch={spec['t4_pair_batch']}"
        )

with open(os.path.join(OUT_ROOT, "execution_plan.json"), "w") as f:
    json.dump(
        {
            "experiment_version": EXPERIMENT_VERSION,
            "config": asdict(cfg),
            "gpu_names": gpu_names,
            "gpu_mem_gib": gpu_mem_gib,
            "model_specs": MODEL_SPECS,
        },
        f,
        indent=2,
    )


In [ ]:

# ============================================================
# HF TOKEN + T4x2 RESOURCE POLICY
# ============================================================

HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

def cpu_available_gib():
    try:
        import psutil
        return psutil.virtual_memory().available / 2**30
    except Exception:
        return 16.0

def model_weight_gib(params_b, dtype_bytes=2):
    return params_b * 1e9 * dtype_bytes / 2**30

def t4_max_memory(extra_cpu_gib=None):
    # Reserve about 1 GiB on each T4 for activations/workspaces.
    mm = {}

    for i in range(torch.cuda.device_count()):
        total_mib = int(
            torch.cuda.get_device_properties(i).total_memory / 2**20
        )
        usable_mib = max(4096, total_mib - cfg.gpu_reserve_mib)
        mm[i] = f"{usable_mib}MiB"

    if cfg.allow_cpu_offload_14b:
        available = max(4, int(cpu_available_gib() * 0.60))
        if extra_cpu_gib is not None:
            available = max(available, int(extra_cpu_gib))
        available = min(available, cfg.cpu_offload_cap_gib)
        mm["cpu"] = f"{available}GiB"

    return mm

def resource_plan(spec):
    p = spec["params_b"]
    weight_gib = model_weight_gib(p)

    if p <= 15.5:
        return {
            "attempt": True,
            "reason": "T4x2 full-precision target",
            "pair_batch": int(spec.get("t4_pair_batch", 1)),
            "weight_gib": weight_gib,
            "device_map": "balanced",
        }

    return {
        "attempt": False,
        "reason": (
            f"FP16 weights ~{weight_gib:.1f} GiB. "
            "Disabled on 2xT4; run later on >=64 GB aggregate VRAM."
        ),
        "pair_batch": 1,
        "weight_gib": weight_gib,
        "device_map": "balanced",
    }

print("CPU available:", round(cpu_available_gib(), 2), "GiB")
print("T4 max_memory map:", t4_max_memory())

print("\nResource plan:")
for spec in MODEL_SPECS:
    if spec["enabled"]:
        print(spec["model_id"], "=>", resource_plan(spec))


In [ ]:

# ============================================================
# FRESH DETERMINISTIC C4 DOCUMENT POOL
# ============================================================

RAW_PATH = os.path.join(OUT_ROOT, "frozen_c4_raw_pool.jsonl")

def deterministic_c4_pool(n_docs, min_chars=2400):
    rows = []
    source_used = "allenai/c4 validation"

    try:
        ds = load_dataset(
            "allenai/c4",
            "en",
            split="validation",
            streaming=True,
            cache_dir=DATA_CACHE,
        )

        for ex in tqdm(ds, desc="Selecting deterministic C4 docs"):
            text = ex.get("text", "")
            if len(text) < min_chars:
                continue

            h = hashlib.sha256(text.encode("utf-8")).hexdigest()

            # Deterministic thinning: independent of Python RNG.
            if int(h[:8], 16) % 11 != 0:
                continue

            rows.append({"sha256": h, "text": text})
            if len(rows) >= n_docs:
                break

    except Exception as e:
        # Notebook remains runnable, but the fallback is NOT a valid fresh-C4 confirmation.
        print("WARNING: C4 streaming failed:", repr(e))
        print("Falling back to WikiText so the pipeline can still be debugged.")
        source_used = "WIKITEXT_FALLBACK_NOT_CONFIRMATORY"
        ds = load_dataset(
            "Salesforce/wikitext",
            "wikitext-103-raw-v1",
            split="test",
            cache_dir=DATA_CACHE,
        )
        text = "\n".join(x for x in ds["text"] if x and len(x) > 10)
        chunks = [text[i:i+5000] for i in range(0, len(text)-5000, 5000)]
        for chunk in chunks:
            h = hashlib.sha256(chunk.encode("utf-8")).hexdigest()
            rows.append({"sha256": h, "text": chunk})
            if len(rows) >= n_docs:
                break

    assert len(rows) >= cfg.n_contexts_core, "Could not obtain enough long documents."
    return rows, source_used

if cfg.resume and os.path.exists(RAW_PATH):
    raw_pool = [json.loads(line) for line in open(RAW_PATH, "r", encoding="utf-8")]
    corpus_source = "REUSED_FROZEN_POOL"
    print("Reusing existing frozen raw pool:", RAW_PATH)
else:
    raw_pool, corpus_source = deterministic_c4_pool(cfg.raw_pool_size)
    with open(RAW_PATH, "w", encoding="utf-8") as f:
        for row in raw_pool:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

pool_hash = hashlib.sha256(
    "".join(r["sha256"] for r in raw_pool).encode("utf-8")
).hexdigest()

print("Corpus source:", corpus_source)
print("Raw docs:", len(raw_pool))
print("Pool SHA256:", pool_hash)

with open(os.path.join(OUT_ROOT, "frozen_config.json"), "w") as f:
    json.dump(
        {
            "experiment_version": EXPERIMENT_VERSION,
            "config": asdict(cfg),
            "pool_hash": pool_hash,
            "corpus_source": corpus_source,
            "model_specs": MODEL_SPECS,
        },
        f,
        indent=2,
    )


In [ ]:

# ============================================================
# GENERIC MODEL / MODULE HELPERS
# ============================================================

def sanitize(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)

def get_by_path(obj, path):
    cur = obj
    for part in path.split("."):
        cur = getattr(cur, part)
    return cur

def resolve_layers(m):
    candidates = [
        "model.layers",
        "model.model.layers",
        "model.language_model.layers",
        "model.language_model.model.layers",
        "language_model.model.layers",
        "transformer.h",
    ]
    for path in candidates:
        try:
            layers = get_by_path(m, path)
            if hasattr(layers, "__len__") and len(layers) > 0:
                return layers, path
        except Exception:
            pass
    raise RuntimeError("Could not resolve transformer block list.")

def resolve_module(block, intervention):
    if intervention == "mlp":
        names = ["mlp", "feed_forward", "ffn", "feed_forward_layer"]
        substrings = ["mlp", "feed", "ffn"]
    elif intervention == "attn":
        names = ["self_attn", "attn", "attention"]
        substrings = ["attn", "attention"]
    else:
        raise ValueError(intervention)

    for name in names:
        if hasattr(block, name):
            return getattr(block, name), name

    for name, module in block.named_children():
        lname = name.lower()
        if any(s in lname for s in substrings):
            return module, name

    raise RuntimeError(f"Could not resolve {intervention} module in block.")

def model_input_device(m):
    for path in [
        "model.embed_tokens",
        "model.model.embed_tokens",
        "model.language_model.embed_tokens",
        "model.language_model.model.embed_tokens",
        "transformer.wte",
    ]:
        try:
            emb = get_by_path(m, path)
            return next(emb.parameters()).device
        except Exception:
            pass
    return next(m.parameters()).device

def unwrap_first_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, (tuple, list)):
        for y in x:
            if torch.is_tensor(y):
                return y
    raise TypeError(f"No tensor found in output type {type(x)}")

def replace_first_tensor(container, new_tensor):
    if torch.is_tensor(container):
        return new_tensor

    if isinstance(container, tuple):
        out = list(container)
        for idx, y in enumerate(out):
            if torch.is_tensor(y):
                out[idx] = new_tensor
                return tuple(out)

    if isinstance(container, list):
        out = list(container)
        for idx, y in enumerate(out):
            if torch.is_tensor(y):
                out[idx] = new_tensor
                return out

    raise TypeError(f"Could not replace tensor in output type {type(container)}")

def tokenize_pool(tokenizer, raw_pool, n, total_tokens):
    ids_list, hashes = [], []
    for row in raw_pool:
        ids = tokenizer(
            row["text"],
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        if len(ids) < total_tokens:
            continue

        ids_list.append(torch.tensor(ids[:total_tokens], dtype=torch.long))
        hashes.append(row["sha256"])

        if len(ids_list) >= n:
            break

    if len(ids_list) < n:
        raise RuntimeError(
            f"Only {len(ids_list)} documents had >= {total_tokens} tokens; need {n}."
        )

    return ids_list, hashes

def spearman(x, y):
    r = spearmanr(np.asarray(x), np.asarray(y), nan_policy="omit")
    return float(r.statistic), float(r.pvalue)

def kl_native_to_patch(z_native, z_patch):
    logp = F.log_softmax(z_native.float(), dim=-1)
    p = logp.exp()
    logq = F.log_softmax(z_patch.float(), dim=-1)
    k = torch.sum(p.unsqueeze(0) * (logp.unsqueeze(0) - logq), dim=-1)
    return torch.clamp(k, min=0.0)

def kl_per_position(z_native, z_patch):
    # z_native: [H,V], z_patch: [B,H,V]
    logp = F.log_softmax(z_native.float(), dim=-1)
    p = logp.exp()
    logq = F.log_softmax(z_patch.float(), dim=-1)
    k = torch.sum(
        p.unsqueeze(0) * (logp.unsqueeze(0) - logq),
        dim=-1,
    )
    return torch.clamp(k, min=0.0)

def fisher_q_from_fd(z_native, z_plus, z_minus, eps):
    dz = (z_plus.float() - z_minus.float()) / (2.0 * eps)
    p = F.softmax(z_native.float(), dim=-1).unsqueeze(0)
    mean = torch.sum(p * dz, dim=-1)
    second = torch.sum(p * dz.square(), dim=-1)
    var = torch.clamp(second - mean.square(), min=0.0)
    return 0.5 * var

def alternating_two_way_residual(frame, value_col, n_iter=100):
    y = frame[value_col].astype(float).to_numpy().copy()
    hosts = frame["host"].to_numpy()
    donors = frame["donor"].to_numpy()
    res = y.copy()

    for _ in range(n_iter):
        for h in np.unique(hosts):
            idx = hosts == h
            res[idx] -= np.mean(res[idx])
        for d in np.unique(donors):
            idx = donors == d
            res[idx] -= np.mean(res[idx])

    return res


In [ ]:

# ============================================================
# TARGET CONTEXT — captures/patches one chosen layer/module
# ============================================================

class TargetContext:
    def __init__(
        self,
        model,
        tokenizer,
        token_ids_core,
        token_ids_full,
        layer_idx,
        intervention,
        pair_batch,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.core_ids = token_ids_core
        self.full_ids = token_ids_full
        self.layer_idx = layer_idx
        self.intervention = intervention
        self.pair_batch = pair_batch

        self.layers, self.layers_path = resolve_layers(model)
        self.block = self.layers[layer_idx]
        self.module, self.module_name = resolve_module(self.block, intervention)
        self.input_device = model_input_device(model)
        self.module_device = next(self.module.parameters()).device
        self.patch_pos = cfg.core_tokens - 1

        self.M = None
        self.U = None
        self.Z0 = None
        self.M_pca = None

    def capture_one(self, ids_1d):
        cap = {}

        def pre_hook(module, inputs):
            x = unwrap_first_tensor(inputs)
            cap["m"] = x[:, self.patch_pos, :].detach().float().cpu()

        def out_hook(module, inputs, output):
            y = unwrap_first_tensor(output)
            cap["u"] = y[:, self.patch_pos, :].detach().float().cpu()

        h1 = self.module.register_forward_pre_hook(pre_hook)
        h2 = self.module.register_forward_hook(out_hook)
        try:
            out = self.model(
                input_ids=ids_1d.unsqueeze(0).to(self.input_device),
                use_cache=False,
                return_dict=True,
            )
            z = out.logits[:, self.patch_pos, :].detach().float().cpu()
        finally:
            h1.remove()
            h2.remove()

        return cap["m"][0], cap["u"][0], z[0]

    def capture_native(self):
        ms, us, zs = [], [], []
        for ids in tqdm(self.core_ids, desc=f"Native {self.intervention} L{self.layer_idx}"):
            m, u, z = self.capture_one(ids)
            ms.append(m)
            us.append(u)
            zs.append(z)

        self.M = torch.stack(ms)
        self.U = torch.stack(us)
        self.Z0 = torch.stack(zs)

        pca_dim = min(32, len(self.core_ids)-1, self.M.shape[1])
        pca = PCA(n_components=pca_dim, random_state=cfg.seed)
        self.M_pca = torch.tensor(
            pca.fit_transform(self.M.numpy()),
            dtype=torch.float32,
        )

    def patched_logits(self, host, replacement_batch):
        ids = self.core_ids[host]
        B = replacement_batch.shape[0]
        batch_ids = ids.unsqueeze(0).repeat(B, 1).to(self.input_device)

        def hook(module, inputs, output):
            y = unwrap_first_tensor(output)
            rep = replacement_batch.to(device=y.device, dtype=y.dtype)
            y2 = y.clone()
            y2[:, self.patch_pos, :] = rep
            return replace_first_tensor(output, y2)

        h = self.module.register_forward_hook(hook)
        try:
            out = self.model(
                input_ids=batch_ids,
                use_cache=False,
                return_dict=True,
            )
            z = out.logits[:, self.patch_pos, :].detach().float().cpu()
        finally:
            h.remove()

        return z

    def q_for_delta(self, host, delta_batch, eps):
        u = self.U[host].unsqueeze(0)
        zp = self.patched_logits(host, u + eps * delta_batch)
        zm = self.patched_logits(host, u - eps * delta_batch)
        return fisher_q_from_fd(self.Z0[host], zp, zm, eps)

    def source_metrics(self, host, donors):
        js = torch.tensor(donors, dtype=torch.long)
        mi = self.M[host].unsqueeze(0)
        ui = self.U[host].unsqueeze(0)
        mj = self.M[js]
        uj = self.U[js]

        dm = mj - mi
        du = uj - ui

        dm_norm = torch.linalg.norm(dm, dim=-1)
        du_norm = torch.linalg.norm(du, dim=-1)
        dist = torch.linalg.norm(
            self.M_pca[js] - self.M_pca[host].unsqueeze(0),
            dim=-1,
        )

        N = torch.full_like(du_norm, float("nan"))
        A = torch.full_like(du_norm, float("nan"))

        # Safe direct MLP-only tangent calculation.
        if self.intervention == "mlp":
            ep = cfg.mlp_source_eps
            mp = (mi + ep * dm).to(self.module_device, dtype=MODEL_DTYPE).unsqueeze(1)
            mm = (mi - ep * dm).to(self.module_device, dtype=MODEL_DTYPE).unsqueeze(1)

            yp = unwrap_first_tensor(self.module(mp))[:, 0, :].detach().float().cpu()
            ym = unwrap_first_tensor(self.module(mm))[:, 0, :].detach().float().cpu()
            tangent = (yp - ym) / (2.0 * ep)

            N = torch.linalg.norm(du - tangent, dim=-1) / torch.clamp(du_norm, min=1e-12)
            A = F.cosine_similarity(tangent, du, dim=-1, eps=1e-12)

        return dict(
            dm_norm=dm_norm,
            du_norm=du_norm,
            source_N=N,
            source_A=A,
            state_dist_pca=dist,
        )

    def self_patch_check(self):
        z = self.patched_logits(0, self.U[0:1])
        k = float(kl_native_to_patch(self.Z0[0], z)[0])
        if k >= 5e-4:
            raise RuntimeError(f"Self-patch KL too large: {k}")
        return k

    def native_future_logits(self, host, horizon):
        ids = self.full_ids[host]
        out = self.model(
            input_ids=ids.unsqueeze(0).to(self.input_device),
            use_cache=False,
            return_dict=True,
        )
        return (
            out.logits[
                0,
                self.patch_pos:self.patch_pos+horizon,
                :
            ]
            .detach()
            .float()
            .cpu()
        )

    def patched_future_logits(self, host, replacement_batch, horizon):
        ids = self.full_ids[host]
        B = replacement_batch.shape[0]
        batch_ids = ids.unsqueeze(0).repeat(B, 1).to(self.input_device)

        def hook(module, inputs, output):
            y = unwrap_first_tensor(output)
            rep = replacement_batch.to(device=y.device, dtype=y.dtype)
            y2 = y.clone()
            y2[:, self.patch_pos, :] = rep
            return replace_first_tensor(output, y2)

        h = self.module.register_forward_hook(hook)
        try:
            out = self.model(
                input_ids=batch_ids,
                use_cache=False,
                return_dict=True,
            )
            z = (
                out.logits[
                    :,
                    self.patch_pos:self.patch_pos+horizon,
                    :
                ]
                .detach()
                .float()
                .cpu()
            )
        finally:
            h.remove()

        return z


In [ ]:

# ============================================================
# CORE 64 x 64 MATRIX
# ============================================================

def run_core_matrix(ctx, target_dir):
    rows = []
    n = len(ctx.core_ids)

    for i in tqdm(range(n), desc="Crossed hosts"):
        donors = [j for j in range(n) if j != i]

        for s in range(0, len(donors), ctx.pair_batch):
            js = donors[s:s+ctx.pair_batch]
            jt = torch.tensor(js, dtype=torch.long)

            ui = ctx.U[i].unsqueeze(0)
            uj = ctx.U[jt]
            delta = uj - ui

            zfull = ctx.patched_logits(i, uj)
            kval = kl_native_to_patch(ctx.Z0[i], zfull)
            qval = ctx.q_for_delta(i, delta, cfg.stable_eps)
            flips = torch.argmax(zfull, dim=-1) != int(torch.argmax(ctx.Z0[i]))
            sm = ctx.source_metrics(i, js)

            for b, j in enumerate(js):
                rows.append({
                    "host": i,
                    "donor": j,
                    "kl": float(kval[b]),
                    "q_local": float(qval[b]),
                    "top1_flip": bool(flips[b]),
                    "dm_norm": float(sm["dm_norm"][b]),
                    "du_norm": float(sm["du_norm"][b]),
                    "source_N": float(sm["source_N"][b]),
                    "source_A": float(sm["source_A"][b]),
                    "state_dist_pca": float(sm["state_dist_pca"][b]),
                })

        torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(target_dir, "pairs.csv"), index=False)
    return df


In [ ]:

# ============================================================
# CORE ANALYSES: H2, H3, held-out ranking, local-bank selection
# ============================================================

def two_way_bootstrap(df, reps, seed):
    rng = np.random.default_rng(seed)
    hosts = np.array(sorted(df["host"].unique()))
    donors = np.array(sorted(df["donor"].unique()))
    lookup = {(int(r.host), int(r.donor)): r for r in df.itertuples(index=False)}
    vals = []

    for _ in range(reps):
        hs = rng.choice(hosts, size=len(hosts), replace=True)
        ds = rng.choice(donors, size=len(donors), replace=True)

        recs = []
        for hi, h in enumerate(hs):
            for di, d in enumerate(ds):
                key = (int(h), int(d))
                if key not in lookup:
                    continue
                r = lookup[key]
                recs.append({
                    "host": hi,
                    "donor": di,
                    "log_kl": float(np.log1p(r.kl)),
                    "log_q": float(np.log1p(r.q_local)),
                })

        b = pd.DataFrame(recs)
        if len(b) < 100:
            continue

        kr = alternating_two_way_residual(b, "log_kl", 50)
        qr = alternating_two_way_residual(b, "log_q", 50)
        rr, _ = spearman(qr, kr)
        if np.isfinite(rr):
            vals.append(rr)

    arr = np.asarray(vals)
    return arr, np.quantile(arr, [0.025, 0.5, 0.975])

def strict_loo(frame, group_col, source_features):
    q_features = source_features + ["log_q"]
    out = []

    for gval in sorted(frame[group_col].unique()):
        tr = frame[frame[group_col] != gval]
        te = frame[frame[group_col] == gval]

        row = {group_col: gval}

        for label, features in [("source", source_features), ("plus_q", q_features)]:
            Xtr = tr[features].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
            Xte = te[features].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
            ytr = tr["log_kl"].to_numpy()
            yte = te["log_kl"].to_numpy()

            pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            pipe.fit(Xtr, ytr)
            pred = pipe.predict(Xte)
            rr, _ = spearman(pred, yte)
            row[f"{label}_rho"] = rr

        row["gain"] = row["plus_q_rho"] - row["source_rho"]
        out.append(row)

    return pd.DataFrame(out)

def selection_curves(df, candidate_sizes):
    rows = []

    for M in candidate_sizes:
        chosen = []

        for host, g in df.groupby("host"):
            bank = g.nsmallest(min(M, len(g)), "state_dist_pca")

            idx_1nn = bank["state_dist_pca"].idxmin()
            idx_q = bank["q_local"].idxmin()
            idx_norm = bank["du_norm"].idxmin()
            idx_oracle = bank["kl"].idxmin()

            chosen.append({
                "host": host,
                "kl_1nn": float(df.loc[idx_1nn, "kl"]),
                "kl_q": float(df.loc[idx_q, "kl"]),
                "kl_norm": float(df.loc[idx_norm, "kl"]),
                "kl_oracle": float(df.loc[idx_oracle, "kl"]),
            })

        z = pd.DataFrame(chosen)
        wins = int((z["kl_q"] < z["kl_1nn"]).sum())
        losses = int((z["kl_q"] > z["kl_1nn"]).sum())
        ties = len(z) - wins - losses

        med_1nn = float(z["kl_1nn"].median())
        med_q = float(z["kl_q"].median())
        reduction = (med_1nn - med_q) / max(med_1nn, 1e-12)

        try:
            wp = float(
                wilcoxon(
                    z["kl_q"],
                    z["kl_1nn"],
                    alternative="less",
                ).pvalue
            )
        except Exception:
            wp = float("nan")

        rows.append({
            "M": M,
            "median_kl_1nn": med_1nn,
            "median_kl_q": med_q,
            "median_reduction": reduction,
            "wins": wins,
            "losses": losses,
            "ties": ties,
            "wilcoxon_p": wp,
        })

    return pd.DataFrame(rows)

def analyze_core(df, target_dir, intervention):
    d = df.copy()
    d["log_kl"] = np.log1p(d["kl"])
    d["log_q"] = np.log1p(d["q_local"])
    d["log_du"] = np.log1p(d["du_norm"])
    d["log_dm"] = np.log1p(d["dm_norm"])

    raw_rho, raw_p = spearman(d["log_q"], d["log_kl"])

    d["kl_resid"] = alternating_two_way_residual(d, "log_kl", 100)
    d["q_resid"] = alternating_two_way_residual(d, "log_q", 100)
    resid_rho, resid_p = spearman(d["q_resid"], d["kl_resid"])

    boot, boot_ci = two_way_bootstrap(
        d,
        cfg.bootstrap_reps,
        cfg.seed + 123,
    )
    pd.DataFrame({"rho": boot}).to_csv(
        os.path.join(target_dir, "two_way_bootstrap.csv"),
        index=False,
    )

    rev = d[["host", "donor", "kl"]].rename(
        columns={"host": "donor", "donor": "host", "kl": "kl_reverse"}
    )
    sym = d.merge(rev, on=["host", "donor"], how="inner")
    reverse_rho, reverse_p = spearman(
        np.log1p(sym["kl"]),
        np.log1p(sym["kl_reverse"]),
    )
    ratio = np.maximum(
        (sym["kl"] + 1e-8) / (sym["kl_reverse"] + 1e-8),
        (sym["kl_reverse"] + 1e-8) / (sym["kl"] + 1e-8),
    )

    if intervention == "mlp":
        source_features = [
            "log_du",
            "log_dm",
            "source_N",
            "source_A",
            "state_dist_pca",
        ]
    else:
        source_features = [
            "log_du",
            "log_dm",
            "state_dist_pca",
        ]

    loo_h = strict_loo(d, "host", source_features)
    loo_d = strict_loo(d, "donor", source_features)
    loo_h.to_csv(os.path.join(target_dir, "strict_loo_host.csv"), index=False)
    loo_d.to_csv(os.path.join(target_dir, "strict_loo_donor.csv"), index=False)

    sel = selection_curves(d, cfg.candidate_sizes)
    sel.to_csv(os.path.join(target_dir, "selection_curves.csv"), index=False)

    return d, {
        "raw_q_rho": raw_rho,
        "raw_q_p": raw_p,
        "resid_q_rho": resid_rho,
        "resid_q_p": resid_p,
        "bootstrap_ci_low": float(boot_ci[0]),
        "bootstrap_ci_median": float(boot_ci[1]),
        "bootstrap_ci_high": float(boot_ci[2]),
        "reverse_rho": reverse_rho,
        "reverse_p": reverse_p,
        "median_directional_ratio": float(np.median(ratio)),
        "strict_host_gain_median": float(loo_h["gain"].median()),
        "strict_host_gain_positive": int((loo_h["gain"] > 0).sum()),
        "strict_donor_gain_median": float(loo_d["gain"].median()),
        "strict_donor_gain_positive": int((loo_d["gain"] > 0).sum()),
        "selection_best_reduction": float(sel["median_reduction"].max()),
        "selection_best_p": float(
            sel.loc[sel["median_reduction"].idxmax(), "wilcoxon_p"]
        ),
    }


In [ ]:

# ============================================================
# EXTRA HYPOTHESIS TESTS AT THE FROZEN HERO DEPTH
# H1/H4/H5/H11/H13
# ============================================================

def run_epsilon_audit(ctx, df, target_dir):
    sample = df.sample(
        n=min(cfg.eps_pairs, len(df)),
        random_state=cfg.seed + 7,
    )[["host", "donor", "kl"]].reset_index(drop=True)

    groups = defaultdict(list)
    for ridx, row in sample.iterrows():
        i, j = int(row.host), int(row.donor)
        groups[i].append((ridx, ctx.U[j] - ctx.U[i]))

    vals = {eps: np.full(len(sample), np.nan) for eps in cfg.eps_grid}

    for eps in cfg.eps_grid:
        for i, tasks in tqdm(groups.items(), desc=f"eps={eps}"):
            for s in range(0, len(tasks), ctx.pair_batch):
                bt = tasks[s:s+ctx.pair_batch]
                deltas = torch.stack([x[1] for x in bt])
                qv = ctx.q_for_delta(i, deltas, eps)
                for b, (ridx, _) in enumerate(bt):
                    vals[eps][ridx] = float(qv[b])

    out = sample.copy()
    for eps in cfg.eps_grid:
        out[f"q_{eps}"] = vals[eps]

    corr_rows = []
    eps_list = list(cfg.eps_grid)

    for a in range(len(eps_list)):
        for b in range(a+1, len(eps_list)):
            ea, eb = eps_list[a], eps_list[b]
            rr, _ = spearman(
                np.log1p(out[f"q_{ea}"]),
                np.log1p(out[f"q_{eb}"]),
            )
            corr_rows.append({"eps_a": ea, "eps_b": eb, "spearman": rr})

    for eps in eps_list:
        rr, _ = spearman(np.log1p(out[f"q_{eps}"]), np.log1p(out["kl"]))
        corr_rows.append({"eps_a": eps, "eps_b": "KL_full", "spearman": rr})

    corr = pd.DataFrame(corr_rows)
    out.to_csv(os.path.join(target_dir, "epsilon_pairs.csv"), index=False)
    corr.to_csv(os.path.join(target_dir, "epsilon_summary.csv"), index=False)

    cross = corr[corr["eps_b"] != "KL_full"]["spearman"].astype(float)
    return {
        "eps_min_cross_rho": float(cross.min()),
        "eps_stable": bool(float(cross.min()) >= cfg.eps_rank_gate),
    }

def run_wrong_receiver(ctx, df, target_dir):
    sample = df.sample(
        n=min(cfg.wrong_pairs, len(df)),
        random_state=cfg.seed + 99,
    ).reset_index(drop=True)

    rng = np.random.default_rng(cfg.seed + 999)
    tasks = defaultdict(list)

    for ridx, row in sample.iterrows():
        i, j = int(row.host), int(row.donor)
        candidates = [k for k in range(len(ctx.core_ids)) if k not in {i, j}]
        ks = rng.choice(
            candidates,
            size=min(cfg.wrong_receivers_per_pair, len(candidates)),
            replace=False,
        )
        delta = ctx.U[j] - ctx.U[i]
        for k in ks:
            tasks[int(k)].append((ridx, delta))

    records = []

    for k, items in tqdm(tasks.items(), desc="wrong receivers"):
        for s in range(0, len(items), ctx.pair_batch):
            bt = items[s:s+ctx.pair_batch]
            deltas = torch.stack([x[1] for x in bt])
            qv = ctx.q_for_delta(k, deltas, cfg.stable_eps)

            for b, (ridx, _) in enumerate(bt):
                row = sample.iloc[ridx]
                records.append({
                    "pair_row": ridx,
                    "orig_host": int(row.host),
                    "orig_donor": int(row.donor),
                    "wrong_host": k,
                    "kl_original": float(row.kl),
                    "q_correct": float(row.q_local),
                    "q_wrong": float(qv[b]),
                })

    wr = pd.DataFrame(records)
    unique = wr.drop_duplicates("pair_row")

    rc, _ = spearman(np.log1p(unique["q_correct"]), np.log1p(unique["kl_original"]))
    rw, _ = spearman(np.log1p(wr["q_wrong"]), np.log1p(wr["kl_original"]))

    agg = (
        wr.groupby("pair_row")
        .agg(
            kl_original=("kl_original", "first"),
            q_correct=("q_correct", "first"),
            q_wrong_mean=("q_wrong", "mean"),
            q_wrong_median=("q_wrong", "median"),
        )
        .reset_index()
    )
    rmean, _ = spearman(np.log1p(agg["q_wrong_mean"]), np.log1p(agg["kl_original"]))
    rmed, _ = spearman(np.log1p(agg["q_wrong_median"]), np.log1p(agg["kl_original"]))

    wr.to_csv(os.path.join(target_dir, "wrong_receiver.csv"), index=False)
    agg.to_csv(os.path.join(target_dir, "wrong_receiver_aggregated.csv"), index=False)

    return {
        "correct_receiver_rho": rc,
        "wrong_receiver_rho": rw,
        "wrong_receiver_mean_rho": rmean,
        "wrong_receiver_median_rho": rmed,
        "receiver_gap": rc - rw,
        "receiver_specificity_pass": bool(
            rc >= cfg.correct_receiver_rho_gate and
            (rc-rw) >= cfg.receiver_gap_gate
        ),
    }

def run_dose_curves(ctx, df, target_dir):
    sample = df.sample(
        n=min(cfg.dose_pairs, len(df)),
        random_state=cfg.seed + 777,
    )[["host", "donor", "kl", "q_local"]].reset_index(drop=True)

    groups = defaultdict(list)
    for ridx, row in sample.iterrows():
        i, j = int(row.host), int(row.donor)
        groups[i].append((ridx, j, ctx.U[j]-ctx.U[i]))

    dose_vals = {
        lam: np.full(len(sample), np.nan)
        for lam in cfg.doses
    }

    dose_vals[1.0] = sample["kl"].to_numpy(float).copy()

    for lam in cfg.doses:
        if float(lam) == 1.0:
            continue

        for i, tasks in tqdm(groups.items(), desc=f"dose={lam}"):
            for s in range(0, len(tasks), ctx.pair_batch):
                bt = tasks[s:s+ctx.pair_batch]
                deltas = torch.stack([x[2] for x in bt])
                replacement = ctx.U[i].unsqueeze(0) + lam * deltas
                z = ctx.patched_logits(i, replacement)
                kval = kl_native_to_patch(ctx.Z0[i], z)

                for b, (ridx, _, _) in enumerate(bt):
                    dose_vals[lam][ridx] = float(kval[b])

    out = sample.copy()
    for lam in cfg.doses:
        out[f"kl_{lam}"] = dose_vals[lam]

    corr_rows = []
    for lam in cfg.doses:
        if float(lam) == 1.0:
            continue
        rr, _ = spearman(np.log1p(out[f"kl_{lam}"]), np.log1p(out["kl_1.0"]))
        corr_rows.append({"dose": lam, "rho_to_full": rr})

    tiny = 1e-10
    out["nonlinearity_025_to_1"] = np.log(
        (out["kl_1.0"] + tiny) /
        (16.0*out["kl_0.25"] + tiny)
    )
    out["rank_K1"] = out["kl_1.0"].rank(pct=True)
    out["rank_Q"] = out["q_local"].rank(pct=True)
    out["local_rank_error"] = (out["rank_K1"] - out["rank_Q"]).abs()

    rnl, pnl = spearman(
        np.abs(out["nonlinearity_025_to_1"]),
        out["local_rank_error"],
    )

    out.to_csv(os.path.join(target_dir, "dose_curves.csv"), index=False)
    pd.DataFrame(corr_rows).to_csv(
        os.path.join(target_dir, "dose_correlations.csv"),
        index=False,
    )

    return {
        "dose_025_to_full_rho": float(
            pd.DataFrame(corr_rows).set_index("dose").loc[0.25, "rho_to_full"]
        ),
        "nonlinearity_failure_rho": rnl,
        "nonlinearity_failure_p": pnl,
    }

def run_future_propagation(ctx, df, target_dir):
    if ctx.full_ids is None:
        return {"future_available": False}

    sample = df.sample(
        n=min(cfg.future_pairs, len(df)),
        random_state=cfg.seed + 4242,
    )[["host", "donor", "q_local"]].reset_index(drop=True)

    groups = defaultdict(list)
    for ridx, row in sample.iterrows():
        groups[int(row.host)].append((ridx, int(row.donor)))

    horizons = list(cfg.future_horizons)
    max_h = max(horizons)

    records = []

    for host, items in tqdm(groups.items(), desc="future propagation"):
        native = ctx.native_future_logits(host, max_h)

        for s in range(0, len(items), cfg.future_batch):
            bt = items[s:s+cfg.future_batch]
            replacements = torch.stack([ctx.U[j] for _, j in bt])
            patch = ctx.patched_future_logits(host, replacements, max_h)
            kpos = kl_per_position(native, patch)

            for b, (ridx, donor) in enumerate(bt):
                row = {
                    "pair_row": ridx,
                    "host": host,
                    "donor": donor,
                    "q_local": float(sample.iloc[ridx].q_local),
                }
                for h in horizons:
                    row[f"cum_kl_{h}"] = float(kpos[b, :h].sum())
                records.append(row)

    out = pd.DataFrame(records)
    out.to_csv(os.path.join(target_dir, "future_propagation.csv"), index=False)

    result = {"future_available": True}
    for h in horizons:
        rr, _ = spearman(np.log1p(out["q_local"]), np.log1p(out[f"cum_kl_{h}"]))
        result[f"future_rho_h{h}"] = rr

    return result

def make_control_directions(delta, seed):
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed))

    norm = torch.linalg.norm(delta)
    d_hat = delta / torch.clamp(norm, min=1e-12)

    g = torch.randn(delta.shape, generator=gen, dtype=delta.dtype)
    g_perp = g - torch.dot(g, d_hat) * d_hat
    g_perp = g_perp / torch.clamp(torch.linalg.norm(g_perp), min=1e-12)

    r = torch.randn(delta.shape, generator=gen, dtype=delta.dtype)
    r = r / torch.clamp(torch.linalg.norm(r), min=1e-12)

    dirs = {"natural": delta, "random": r * norm}

    for deg in cfg.rotation_degrees:
        theta = math.radians(deg)
        rot = (
            math.cos(theta) * d_hat +
            math.sin(theta) * g_perp
        ) * norm
        dirs[f"rot{deg}"] = rot

    return dirs

def run_natural_controls(ctx, df, target_dir):
    sample = df.sample(
        n=min(cfg.control_pairs, len(df)),
        random_state=cfg.seed + 616,
    )[["host", "donor"]].reset_index(drop=True)

    rows = []

    for ridx, row in tqdm(sample.iterrows(), total=len(sample), desc="natural controls"):
        i, j = int(row.host), int(row.donor)
        delta = ctx.U[j] - ctx.U[i]
        dirs = make_control_directions(delta, cfg.seed + i*10007 + j*101)

        names = list(dirs)
        vecs = torch.stack([dirs[n] for n in names])

        q_all = []
        k1_all = []
        k025_all = []

        for s in range(0, len(names), ctx.pair_batch):
            chunk = vecs[s:s+ctx.pair_batch]
            qv = ctx.q_for_delta(i, chunk, cfg.stable_eps)

            z1 = ctx.patched_logits(
                i,
                ctx.U[i].unsqueeze(0) + chunk,
            )
            z025 = ctx.patched_logits(
                i,
                ctx.U[i].unsqueeze(0) + 0.25*chunk,
            )

            k1 = kl_native_to_patch(ctx.Z0[i], z1)
            k025 = kl_native_to_patch(ctx.Z0[i], z025)

            q_all.extend(qv.tolist())
            k1_all.extend(k1.tolist())
            k025_all.extend(k025.tolist())

        for name, qv, k1, k025 in zip(names, q_all, k1_all, k025_all):
            nl = math.log((k1 + 1e-10) / (16.0*k025 + 1e-10))
            rows.append({
                "pair_id": ridx,
                "host": i,
                "donor": j,
                "direction_type": name,
                "q_local": qv,
                "kl_025": k025,
                "kl_1": k1,
                "abs_nonlinearity": abs(nl),
                "nonlinearity": nl,
            })

    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(target_dir, "natural_vs_controls.csv"), index=False)

    pivot = out.pivot(index="pair_id", columns="direction_type", values="abs_nonlinearity")
    comparisons = []

    if "natural" in pivot:
        for col in pivot.columns:
            if col == "natural":
                continue
            try:
                p = float(
                    wilcoxon(
                        pivot["natural"],
                        pivot[col],
                        alternative="less",
                    ).pvalue
                )
            except Exception:
                p = float("nan")
            comparisons.append({
                "control": col,
                "median_natural": float(pivot["natural"].median()),
                "median_control": float(pivot[col].median()),
                "p_natural_less": p,
            })

    comp = pd.DataFrame(comparisons)
    comp.to_csv(os.path.join(target_dir, "natural_control_summary.csv"), index=False)

    return {
        "natural_control_tests": comparisons,
    }


In [ ]:

# ============================================================
# ONE TARGET RUNNER
# ============================================================

def target_name(model_id, layer_idx, n_layers, intervention):
    rel = layer_idx / max(n_layers-1, 1)
    return f"{sanitize(model_id)}__{intervention}__L{layer_idx}__r{rel:.3f}"

def run_target(
    model,
    tokenizer,
    model_id,
    token_core,
    token_full,
    layer_idx,
    intervention,
    pair_batch,
    run_extras,
):
    layers, _ = resolve_layers(model)
    tname = target_name(model_id, layer_idx, len(layers), intervention)
    tdir = os.path.join(OUT_ROOT, "targets", tname)
    os.makedirs(tdir, exist_ok=True)

    summary_path = os.path.join(tdir, "summary.json")
    if cfg.resume and os.path.exists(summary_path):
        print("RESUME: already complete:", tname)
        with open(summary_path) as f:
            return json.load(f)

    start = time.time()

    ctx = TargetContext(
        model=model,
        tokenizer=tokenizer,
        token_ids_core=token_core,
        token_ids_full=token_full,
        layer_idx=layer_idx,
        intervention=intervention,
        pair_batch=pair_batch,
    )

    ctx.capture_native()
    self_kl = ctx.self_patch_check()
    print("Self-patch KL:", self_kl)

    df = run_core_matrix(ctx, tdir)
    analyzed, core = analyze_core(df, tdir, intervention)

    summary = {
        "target": tname,
        "model_id": model_id,
        "intervention": intervention,
        "layer_idx": layer_idx,
        "n_layers": len(layers),
        "relative_depth": layer_idx / max(len(layers)-1, 1),
        "n_contexts": len(token_core),
        "pair_batch": pair_batch,
        "self_patch_kl": self_kl,
        **core,
    }

    # Fresh confirmatory gates on every target.
    summary["H2_pair_specific_pass"] = bool(
        summary["resid_q_rho"] >= cfg.residual_rho_gate and
        summary["bootstrap_ci_low"] > 0
    )
    summary["H3_directional_descriptive"] = bool(
        summary["reverse_rho"] < 0.30
    )

    if run_extras:
        eps = run_epsilon_audit(ctx, analyzed, tdir)
        summary.update(eps)

        wrong = run_wrong_receiver(ctx, analyzed, tdir)
        summary.update(wrong)

        dose = run_dose_curves(ctx, analyzed, tdir)
        summary.update(dose)

        future = run_future_propagation(ctx, analyzed, tdir)
        summary.update(future)

        controls = run_natural_controls(ctx, analyzed, tdir)
        summary.update(controls)

        summary["H1_receiver_specificity_pass"] = bool(
            summary.get("receiver_specificity_pass", False) and
            summary.get("eps_stable", False)
        )
        summary["H4_finite_dose_pass"] = bool(
            summary.get("raw_q_rho", -1) >= 0.75 and
            summary.get("dose_025_to_full_rho", -1) >= 0.85
        )
        summary["H11_future_32_pass"] = bool(
            summary.get("future_rho_h32", -1) >= 0.50
        )

    summary["elapsed_minutes"] = (time.time() - start) / 60.0

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    del ctx
    gc.collect()
    torch.cuda.empty_cache()

    return summary


In [ ]:

# ============================================================
# T4x2 MODEL LOADER / UNLOADER
# ============================================================

def _load_once(spec, max_memory, device_map="balanced"):
    tokenizer = AutoTokenizer.from_pretrained(
        spec["model_id"],
        token=HF_TOKEN,
        trust_remote_code=True,
        cache_dir=MODEL_CACHE,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        spec["model_id"],
        token=HF_TOKEN,
        trust_remote_code=True,
        torch_dtype=MODEL_DTYPE,
        device_map=device_map,
        max_memory=max_memory,
        low_cpu_mem_usage=True,
        offload_folder=OFFLOAD_DIR,
        offload_state_dict=True,
        cache_dir=MODEL_CACHE,
    )

    model.eval()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    for p in model.parameters():
        p.requires_grad_(False)

    return model, tokenizer

def load_model_and_tokenizer(spec):
    plan = resource_plan(spec)
    if not plan["attempt"]:
        raise RuntimeError("RESOURCE_SKIP: " + plan["reason"])

    print("\n" + "="*90)
    print("LOADING:", spec["model_id"])
    print("Estimated FP16 weight size:", round(plan["weight_gib"], 2), "GiB")
    print("Requested pair batch:", plan["pair_batch"])
    print("="*90)

    torch.cuda.empty_cache()
    gc.collect()

    attempts = [
        {
            "label": "balanced_t4x2",
            "max_memory": t4_max_memory(),
            "device_map": "balanced",
        }
    ]

    if spec["params_b"] > 10:
        attempts.append({
            "label": "balanced_t4x2_more_cpu",
            "max_memory": t4_max_memory(extra_cpu_gib=16),
            "device_map": "balanced",
        })

    last_error = None

    for attempt_idx, a in enumerate(attempts, start=1):
        model = None
        tokenizer = None
        try:
            print(f"Load attempt {attempt_idx}/{len(attempts)}:", a["label"])
            print("max_memory:", a["max_memory"])

            model, tokenizer = _load_once(
                spec,
                max_memory=a["max_memory"],
                device_map=a["device_map"],
            )

            dmap = getattr(model, "hf_device_map", {})
            disk_parts = [k for k, v in dmap.items() if str(v) == "disk"]
            cpu_parts = [k for k, v in dmap.items() if str(v) == "cpu"]

            if disk_parts:
                del model
                del tokenizer
                gc.collect()
                torch.cuda.empty_cache()
                raise RuntimeError(
                    "RESOURCE_SKIP: model required disk offload, "
                    "which is not acceptable for thousands of interventions."
                )

            print("Loaded successfully.")
            print("CPU-mapped module count:", len(cpu_parts))
            print("GPU allocated:")
            for i in range(torch.cuda.device_count()):
                print(
                    f"  GPU{i}: "
                    f"{torch.cuda.memory_allocated(i)/2**30:.2f} GiB allocated | "
                    f"{torch.cuda.memory_reserved(i)/2**30:.2f} GiB reserved"
                )

            return model, tokenizer, plan

        except Exception as e:
            last_error = e
            print("Load attempt failed:", repr(e)[:1200])

            if model is not None:
                del model
            if tokenizer is not None:
                del tokenizer

            gc.collect()
            torch.cuda.empty_cache()

            msg = str(e).lower()
            retryable = any(
                key in msg
                for key in [
                    "out of memory",
                    "cuda",
                    "memory",
                    "allocate",
                    "offload",
                ]
            )
            if not retryable and "resource_skip" not in msg:
                raise

    raise RuntimeError(
        f"MODEL_LOAD_FAILED_AFTER_RETRY: {spec['model_id']} | {repr(last_error)}"
    )

def unload_model(model, tokenizer):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass

    gc.collect()
    torch.cuda.empty_cache()

    if os.path.exists(OFFLOAD_DIR):
        shutil.rmtree(OFFLOAD_DIR, ignore_errors=True)
        os.makedirs(OFFLOAD_DIR, exist_ok=True)

def clear_model_cache():
    if cfg.delete_model_cache_after_each_model and os.path.exists(MODEL_CACHE):
        shutil.rmtree(MODEL_CACHE, ignore_errors=True)
        os.makedirs(MODEL_CACHE, exist_ok=True)

def write_checkpoint_zip():
    tmp = CHECKPOINT_ZIP + ".tmp"
    if os.path.exists(tmp):
        os.remove(tmp)

    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(OUT_ROOT):
            for fn in files:
                path = os.path.join(root, fn)
                arc = os.path.relpath(path, OUT_ROOT)
                zf.write(path, arcname=arc)

    os.replace(tmp, CHECKPOINT_ZIP)
    print("Rolling checkpoint:", CHECKPOINT_ZIP)


# Run the frozen reproduction

Run the next cell once after the setup and helper cells.

The notebook processes Qwen3-8B first, clears its downloaded model cache, then processes Mistral-7B-v0.3. Each completed target writes a `summary.json` and pair-level CSVs immediately. Attention-output experiments and larger models are intentionally excluded because they are not reported in the workshop manuscript.


In [ ]:

# ============================================================
# RUN THE COMPLETE T4x2 FROZEN EXPERIMENT MATRIX
# ============================================================

all_summaries = []
model_status = []

def run_target_with_batch_retry(
    *,
    model,
    tokenizer,
    model_id,
    token_core,
    token_full,
    layer_idx,
    intervention,
    initial_batch,
    run_extras,
):
    attempts = []
    for b in [initial_batch, max(1, initial_batch//2), 1]:
        if b not in attempts:
            attempts.append(b)

    last_err = None

    for batch in attempts:
        try:
            print(
                f"Target attempt: {model_id} | {intervention} "
                f"| layer={layer_idx} | pair_batch={batch}"
            )
            return run_target(
                model=model,
                tokenizer=tokenizer,
                model_id=model_id,
                token_core=token_core,
                token_full=token_full,
                layer_idx=layer_idx,
                intervention=intervention,
                pair_batch=batch,
                run_extras=run_extras,
            )

        except torch.cuda.OutOfMemoryError as e:
            last_err = e
            print("CUDA OOM at pair_batch", batch, "-> retrying smaller batch.")
            gc.collect()
            torch.cuda.empty_cache()

        except RuntimeError as e:
            msg = str(e).lower()
            if "out of memory" in msg or "cuda error" in msg:
                last_err = e
                print("Runtime memory failure at pair_batch", batch, "-> retrying.")
                gc.collect()
                torch.cuda.empty_cache()
            else:
                raise

    raise RuntimeError(
        f"TARGET_OOM_AFTER_RETRIES: {model_id} {intervention} "
        f"layer={layer_idx} | {repr(last_err)}"
    )

for spec in MODEL_SPECS:
    if not spec["enabled"]:
        continue

    model = None
    tokenizer = None

    try:
        plan = resource_plan(spec)

        if not plan["attempt"]:
            print("\nSKIP:", spec["model_id"], "|", plan["reason"])
            model_status.append({
                "model_id": spec["model_id"],
                "status": "SKIPPED_RESOURCE",
                "detail": plan["reason"],
            })
            continue

        model, tokenizer, plan = load_model_and_tokenizer(spec)

        full_len = cfg.core_tokens + cfg.future_tokens
        token_full, doc_hashes = tokenize_pool(
            tokenizer,
            raw_pool,
            cfg.n_contexts_core,
            full_len,
        )
        token_core = [x[:cfg.core_tokens].clone() for x in token_full]

        model_dir = os.path.join(
            OUT_ROOT,
            "models",
            sanitize(spec["model_id"]),
        )
        os.makedirs(model_dir, exist_ok=True)

        with open(os.path.join(model_dir, "document_hashes.json"), "w") as f:
            json.dump(doc_hashes, f, indent=2)

        layers, layer_path = resolve_layers(model)
        n_layers = len(layers)

        print("\nResolved", n_layers, "layers via", layer_path)

        for rel in spec["depths"]:
            layer_idx = min(
                n_layers - 1,
                max(0, int(round(rel * (n_layers - 1)))),
            )
            run_extras = abs(rel - spec["full_extras_depth"]) < 1e-9

            try:
                s = run_target_with_batch_retry(
                    model=model,
                    tokenizer=tokenizer,
                    model_id=spec["model_id"],
                    token_core=token_core,
                    token_full=token_full,
                    layer_idx=layer_idx,
                    intervention="mlp",
                    initial_batch=plan["pair_batch"],
                    run_extras=run_extras,
                )
                s["role"] = spec["role"]
                s["status"] = "TARGET_COMPLETED"
                all_summaries.append(s)

            except Exception as e:
                print("MLP target failed; recording and continuing:", repr(e))
                all_summaries.append({
                    "model_id": spec["model_id"],
                    "role": spec["role"],
                    "relative_depth_requested": rel,
                    "intervention": "mlp",
                    "status": "TARGET_ERROR",
                    "error": repr(e)[:1500],
                })
                gc.collect()
                torch.cuda.empty_cache()

        if spec.get("run_attention", False):
            rel = spec["full_extras_depth"]
            layer_idx = min(
                n_layers - 1,
                max(0, int(round(rel * (n_layers - 1)))),
            )

            attn_core = token_core[:cfg.n_contexts_attention]
            attn_full = token_full[:cfg.n_contexts_attention]

            try:
                s = run_target_with_batch_retry(
                    model=model,
                    tokenizer=tokenizer,
                    model_id=spec["model_id"],
                    token_core=attn_core,
                    token_full=attn_full,
                    layer_idx=layer_idx,
                    intervention="attn",
                    initial_batch=max(1, min(plan["pair_batch"], 2)),
                    run_extras=True,
                )
                s["role"] = spec["role"] + "_attention"
                s["status"] = "TARGET_COMPLETED"
                all_summaries.append(s)

            except Exception as e:
                print("Attention target failed; recording:", repr(e))
                all_summaries.append({
                    "model_id": spec["model_id"],
                    "role": spec["role"] + "_attention",
                    "relative_depth_requested": rel,
                    "intervention": "attn",
                    "status": "TARGET_ERROR",
                    "error": repr(e)[:1500],
                })
                gc.collect()
                torch.cuda.empty_cache()

        model_status.append({
            "model_id": spec["model_id"],
            "status": "MODEL_COMPLETED",
            "detail": "all configured targets processed",
        })

    except Exception as e:
        msg = str(e)
        status = (
            "SKIPPED_RESOURCE"
            if "RESOURCE_SKIP:" in msg
            else "MODEL_ERROR"
        )

        print("\nMODEL FAILED/SKIPPED:", spec["model_id"], repr(e))

        model_status.append({
            "model_id": spec["model_id"],
            "status": status,
            "detail": repr(e)[:1800],
        })
        traceback.print_exc(limit=1)

    finally:
        if model is not None:
            unload_model(model, tokenizer)

        pd.DataFrame(all_summaries).to_csv(
            os.path.join(OUT_ROOT, "all_target_summaries.csv"),
            index=False,
        )
        pd.DataFrame(model_status).to_csv(
            os.path.join(OUT_ROOT, "model_status.csv"),
            index=False,
        )

        clear_model_cache()
        write_checkpoint_zip()

        print("\nCurrent model status:")
        display(pd.DataFrame(model_status))

print("\nALL CONFIGURED T4x2 MODELS PROCESSED.")
display(pd.DataFrame(model_status))


In [ ]:

# ============================================================
# REPRODUCTION SUMMARY
# ============================================================

summ = pd.DataFrame(all_summaries)
if "status" in summ.columns:
    ok = summ[summ["status"].fillna("TARGET_COMPLETED") == "TARGET_COMPLETED"].copy()
else:
    ok = summ.copy()

cols = [
    "model_id", "intervention", "relative_depth",
    "raw_q_rho", "resid_q_rho", "bootstrap_ci_low", "bootstrap_ci_high",
    "reverse_rho", "correct_receiver_rho", "wrong_receiver_rho",
    "receiver_gap", "strict_host_gain_median", "strict_donor_gain_median",
    "selection_best_reduction", "selection_best_p",
    "dose_025_to_full_rho", "future_rho_h32", "eps_min_cross_rho",
]
cols = [c for c in cols if c in ok.columns]
summary = ok[cols].copy()
summary.to_csv(os.path.join(OUT_ROOT, "reported_target_summary.csv"), index=False)
display(summary)

q = summary[
    (summary["model_id"] == "Qwen/Qwen3-8B") &
    (np.isclose(summary["relative_depth"].astype(float), 0.40, atol=0.02))
]
m = summary[
    (summary["model_id"] == "mistralai/Mistral-7B-v0.3") &
    (np.isclose(summary["relative_depth"].astype(float), 0.3870967742, atol=0.02))
]

assert len(q) == 1 and len(m) == 1, "Primary-depth targets are missing."
assert float(q.iloc[0]["resid_q_rho"]) >= 0.30
assert float(m.iloc[0]["resid_q_rho"]) >= 0.30
assert float(q.iloc[0]["bootstrap_ci_low"]) > 0
assert float(m.iloc[0]["bootstrap_ci_low"]) > 0

print("Fresh-C4 reproduction completed successfully.")


In [ ]:

# ============================================================
# PAPER-READY SUMMARY FIGURES
# ============================================================

import matplotlib.pyplot as plt

if len(hyp):
    plot_df = hyp[
        (hyp["intervention"] == "mlp") &
        hyp["raw_q_rho"].notna()
    ].copy()

    # 1) Q -> KL by depth/model
    fig, ax = plt.subplots(figsize=(9, 5))
    for model_id, g in plot_df.groupby("model_id"):
        g = g.sort_values("relative_depth")
        ax.plot(
            g["relative_depth"],
            g["raw_q_rho"],
            marker="o",
            label=model_id,
        )
    ax.axhline(0.75, linestyle="--", linewidth=1)
    ax.set_xlabel("Relative depth")
    ax.set_ylabel("Spearman rho: Q vs full-dose KL")
    ax.set_title("Receiver-side sensitivity across model depth")
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUT_ROOT, "fig_depth_raw_q.png"),
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    # 2) Pair-specific residual association
    fig, ax = plt.subplots(figsize=(9, 5))
    labels = (
        plot_df["model_id"].str.split("/").str[-1] +
        "@" +
        plot_df["relative_depth"].round(2).astype(str)
    )
    x = np.arange(len(plot_df))
    ax.bar(x, plot_df["resid_q_rho"])
    ax.axhline(cfg.residual_rho_gate, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=65, ha="right", fontsize=7)
    ax.set_ylabel("Two-way residualized Spearman rho")
    ax.set_title("Pair-specific receiver-donor compatibility")
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUT_ROOT, "fig_pair_specific.png"),
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

print("Figures saved in:", OUT_ROOT)


In [ ]:

# ============================================================
# PACKAGE EVERYTHING FOR UPLOAD BACK TO CHATGPT
# ============================================================

zip_path = f"/kaggle/working/{EXPERIMENT_VERSION}_RESULTS.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUT_ROOT):
        for fn in files:
            path = os.path.join(root, fn)
            arc = os.path.relpath(path, OUT_ROOT)
            zf.write(path, arcname=arc)

print("Created:", zip_path)
print("Upload this ZIP back to ChatGPT for the PI audit.")


## Output

The final packaging cell creates a ZIP under `/kaggle/working/NEWINML_FRESH_C4_REPRO_V1_RESULTS.zip`.

For review-time numerical verification, rerunning the GPU notebook is not required. The repository's `analysis/audit_fresh_c4_multibillion.py` script checks the frozen outputs directly on CPU.
